# Percentage Based Balancing

In [3]:
import pandas as pd                                     
                                                        
# Load the dataset                                      
df = pd.read_csv('Train_Set_70_Augmented.csv')          
                                                        
# Convert the column to object/text              
df['sentiment'] = df['sentiment'].astype(object)        
                                                        
# Instantly map all synthetic rows to NEGATIVE          
df.loc[df['is_synthetic'] == True, 'sentiment'] = 'NEGATIVE'                                                
                                                        
# Save it back (overwriting the file)                   
df.to_csv('Train_Set_70_Augmented.csv', index=False,    
encoding='utf-8')                                         
                                                        
print(f"Successfully labeled {(df['is_synthetic'] ==    
True).sum()} synthetic rows as NEGATIVE.")     

Successfully labeled 494 synthetic rows as NEGATIVE.


In [13]:
df_gt = pd.read_csv('Ground_Truth_500.csv')                                                                                                                      
df_train = pd.read_csv('Train_Set_70_Augmented.csv')                                                                                                                        
                                                                                                                                                                            
                                                                                                                                       
df_synthetic_pool = df_train[                                                                                                                                               
    (df_train['is_synthetic'] == True) &                                                                                                                                    
    (df_train['sentiment'] == 'NEGATIVE')                                                                                                                                   
].copy()                                                                                                                                                                    
                                                                                                                                                                            
# Define the absolute hard limits (Total = 214)                                                                                                                          
injection_targets = {                                                                                                                                                       
    "Ambiance and Atmosphere": 53,                                                                                                                                          
    "Facilities and Amenities": 53,                                                                                                                                         
    "Price and Value": 54,                                                                                                                                                  
    "Store Operations and Accessibility": 54                                                                                                                                
}                                                                                                                                                                           
                                                                                                                                                                            
                                                                                                                       
sampled_list = []                                                                                                                                                           
for aspect, quota in injection_targets.items():                                                                                                                             
                                                                                                               
    subset = df_synthetic_pool[                                                                                                                                             
        (df_synthetic_pool['TargetAspect'] == aspect) |                                                                                                                     
        (df_synthetic_pool['TargetAspect'] == aspect.split()[0])                                                                                                            
    ]                                                                                                                                                                       
    assert len(subset) >= quota, f"Not enough synthetic samples for '{aspect}'. Found {len(subset)}, needed {quota}."                                                       
    sampled_list.append(subset.sample(n=quota, random_state=42))                                                                                                            
                                                                                                                                                                            
df_synthetic_sampled = pd.concat(sampled_list, ignore_index=True)                                                                                                           
                                                                                                                                                                            
                                                                                                                                         
df_synthetic_sampled['ground_truth'] = 'NEGATIVE'                                                                                                                           
df_synthetic_sampled['is_tie'] = False                                                                                                                                      
df_synthetic_sampled['vote_breakdown'] = 'SYNTHETIC_GENERATION'

In [14]:
df_augmented = pd.concat([df_gt, df_synthetic_sampled], ignore_index=True)                                                                                                  
output_file = 'Augmented_Ground_Truth.csv'                                                                                                                                  
df_augmented.to_csv(output_file, index=False, encoding='utf-8')                                                                                                             
                                                                                                                                                                            
#Verification metrics                                                                                                                                                   
print(f"Authentic Ground Truth: {len(df_gt)} rows")                                                                                                                         
print(f"Synthetic Rows Injected: {len(df_synthetic_sampled)} rows")                                                                                                         
print(f"Total Augmented Size: {len(df_augmented)} rows")                                                                                                                    
print(f"Synthetic Ratio: {(len(df_synthetic_sampled) / len(df_augmented)) * 100:.2f}%\n")                                                                                   
                                                                                                                                                                            
print("--- Breakdown of Injected Synthetic Rows by Aspect ---")                                                                                                             
print(df_synthetic_sampled['TargetAspect'].value_counts())                                                                                                                  


Authentic Ground Truth: 500 rows
Synthetic Rows Injected: 214 rows
Total Augmented Size: 714 rows
Synthetic Ratio: 29.97%

--- Breakdown of Injected Synthetic Rows by Aspect ---
TargetAspect
Price and Value                       54
Store Operations and Accessibility    54
Ambiance and Atmosphere               53
Facilities and Amenities              53
Name: count, dtype: int64


# Testing